# ACRS — Automated Code Review System
## Exploratory Data Analysis & Model Evaluation Report

**Authors:** Ritwik Kumar, Sristi Banik, Ritul Shekhar | **Supervisor:** Dr. Syed Ismail A. | **SRMIST**

This notebook evaluates the GNN-based Automated Code Review System against a **20-sample labelled benchmark** with known defect annotations.

---

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.patches as mpatches, seaborn as sns
from collections import defaultdict
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 150, 'axes.spines.top': False, 'axes.spines.right': False})
COLORS = {'Bug-Prone':'#FF3B30','Code Smell':'#FF9500','Design Inefficiency':'#AF52DE','Clean':'#34C759','accent':'#0071E3','dark':'#1D1D1F','gray':'#86868B','green':'#34C759'}
with open('benchmark_results.json') as f: results = json.load(f)
print(f"Loaded: {results['benchmark_info']['total_samples']} samples, {results['benchmark_info']['total_ground_truth_issues']} GT issues")

## 1. Overall Performance Metrics
Micro-averaged (per-prediction) and macro-averaged (per-category) precision, recall, and F1.

In [ ]:
o = results['overall_metrics']; bi = results['benchmark_info']
df = pd.DataFrame({'Metric':['Micro P','Micro R','Micro F1','Macro P','Macro R','Macro F1'],
    'Score':[o['micro_precision'],o['micro_recall'],o['micro_f1'],o['macro_precision'],o['macro_recall'],o['macro_f1']]})
fig, axes = plt.subplots(1,2,figsize=(14,5),gridspec_kw={'width_ratios':[2,1]})
bars = axes[0].barh(df['Metric'],df['Score'],color=[COLORS['accent'] if 'Micro' in m else '#AF52DE' for m in df['Metric']],height=0.6,edgecolor='white')
for bar,val in zip(bars,df['Score']): axes[0].text(bar.get_width()+0.02,bar.get_y()+bar.get_height()/2,f'{val*100:.1f}%',va='center',fontweight='bold',fontsize=12)
axes[0].set_xlim(0,1.15); axes[0].set_title('Overall Classification Metrics',fontsize=14,fontweight='bold',pad=15)
axes[0].axvline(x=0.8,color='#34C759',linestyle='--',alpha=0.5,label='80% threshold'); axes[0].legend(fontsize=9)
txt = f"Samples: {bi['total_samples']}\nGT Issues: {bi['total_ground_truth_issues']}\nClean: {bi['clean_samples']}\nDefective: {bi['defective_samples']}\nAvg Latency: {bi['avg_latency_ms']}ms"
axes[1].text(0.1,0.5,txt,transform=axes[1].transAxes,fontsize=13,fontfamily='monospace',va='center',bbox=dict(boxstyle='round,pad=0.8',facecolor='#F5F5F7',edgecolor='#D2D2D7'))
axes[1].axis('off'); axes[1].set_title('Config',fontsize=14,fontweight='bold')
plt.tight_layout(); plt.show()

## 2. Confusion Matrix
Diagonal = correct predictions (green). Off-diagonal = misclassifications (red).

In [ ]:
cm = results['confusion_matrix']; labels = cm['labels']; matrix = np.array(cm['matrix'])
fig, ax = plt.subplots(figsize=(9,7))
mask_d = np.eye(len(labels),dtype=bool); mask_o = ~mask_d
sns.heatmap(matrix,annot=True,fmt='d',cmap=sns.light_palette('#34C759',as_cmap=True),mask=mask_o,xticklabels=labels,yticklabels=labels,ax=ax,annot_kws={'size':18,'weight':'bold'},cbar=False,linewidths=3,linecolor='white',square=True)
sns.heatmap(matrix,annot=True,fmt='d',cmap=sns.light_palette('#FF3B30',as_cmap=True),mask=mask_d,xticklabels=labels,yticklabels=labels,ax=ax,annot_kws={'size':18,'weight':'bold'},cbar=False,linewidths=3,linecolor='white',square=True)
ax.set_xlabel('Predicted',fontsize=13,fontweight='bold'); ax.set_ylabel('Actual',fontsize=13,fontweight='bold')
ax.set_title('Confusion Matrix',fontsize=15,fontweight='bold',pad=20); plt.setp(ax.get_xticklabels(),rotation=20,ha='right')
plt.tight_layout(); plt.show()
print(f"Accuracy: {np.trace(matrix)}/{matrix.sum()} = {np.trace(matrix)/matrix.sum()*100:.1f}%")

## 3. Per-Category Precision, Recall, F1

In [ ]:
pc = results['per_category_metrics']; cats = list(pc.keys())
p_=[pc[c]['precision'] for c in cats]; r_=[pc[c]['recall'] for c in cats]; f_=[pc[c]['f1'] for c in cats]
cc = [COLORS.get(c,COLORS['accent']) for c in cats]
fig, axes = plt.subplots(1,3,figsize=(16,5.5)); x = np.arange(len(cats))
for ax,vals,title in [(axes[0],p_,'Precision'),(axes[1],r_,'Recall'),(axes[2],f_,'F1 Score')]:
    bars = ax.bar(x,vals,0.6,color=cc,edgecolor='white')
    for b,v in zip(bars,vals): ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.02,f'{v*100:.0f}%',ha='center',fontweight='bold',fontsize=11)
    ax.set_ylim(0,1.2); ax.set_xticks(x); ax.set_xticklabels([c.replace(' ','\n') for c in cats],fontsize=9); ax.set_title(title,fontsize=14,fontweight='bold'); ax.axhline(y=0.8,color='#34C759',linestyle='--',alpha=0.4)
plt.suptitle('Per-Category Performance',fontsize=16,fontweight='bold',y=1.02); plt.tight_layout(); plt.show()
for c in cats:
    m=pc[c]; s="Strong" if m['f1']>=0.8 else "Moderate" if m['f1']>=0.5 else "Weak"
    print(f"  {'✅' if s=='Strong' else '⚠️' if s=='Moderate' else '❌'} {c}: F1={m['f1']*100:.1f}% — {s}")

## 4. Per-Pattern Detection Performance

In [ ]:
pp = results['per_pattern_metrics']; pats = list(pp.keys())
pf = [pp[p]['f1'] for p in pats]; pprc = [pp[p]['precision'] for p in pats]; prc = [pp[p]['recall'] for p in pats]; ps = [pp[p]['support'] for p in pats]
si = np.argsort(pf)[::-1]; pats_s = [pats[i].replace('_',' ').title() for i in si]
fig, ax = plt.subplots(figsize=(12,6)); y = np.arange(len(pats_s)); h = 0.25
ax.barh(y+h,[pf[i] for i in si],h,label='F1',color=COLORS['accent'],edgecolor='white')
ax.barh(y,[pprc[i] for i in si],h,label='Precision',color='#5AC8FA',edgecolor='white')
ax.barh(y-h,[prc[i] for i in si],h,label='Recall',color='#34C759',edgecolor='white')
for i2,idx in enumerate(si): ax.text(max(pf[idx],pprc[idx],prc[idx])+0.03,i2,f'n={ps[idx]}',va='center',fontsize=10,color=COLORS['gray'])
ax.set_yticks(y); ax.set_yticklabels(pats_s,fontsize=11); ax.set_xlim(0,1.25); ax.set_title('Detection by Pattern',fontsize=15,fontweight='bold',pad=15); ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

## 5. Graph Structural Analysis
How AST/CFG/DFG edge distributions relate to detection performance.

In [ ]:
ga = results['graph_analysis']; dg = pd.DataFrame(ga)
fig, axes = plt.subplots(2,2,figsize=(14,10))
axes[0,0].scatter(dg['num_nodes'],dg['num_edges'],c=COLORS['accent'],s=dg['gt_count']*80+40,alpha=0.6,edgecolors='white')
axes[0,0].set_xlabel('Nodes'); axes[0,0].set_ylabel('Edges'); axes[0,0].set_title('Graph Size',fontweight='bold')
et=['ast_edges','cfg_edges','dfg_edges']; el=['AST','CFG','DFG']; ec=[COLORS['dark'],COLORS['accent'],'#FF3B30']
tots=[dg[e].sum() for e in et]
axes[0,1].pie(tots,labels=el,colors=ec,autopct='%1.1f%%',startangle=90,wedgeprops={'edgecolor':'white','linewidth':2})
axes[0,1].set_title('Edge Type Distribution',fontweight='bold')
cr = []; 
for _,row in dg.iterrows():
    if row['gt_count']>0: cr.append(row['correct']/row['gt_count'])
    else: cr.append(1.0 if row['false_alarms']==0 else 0.5)
axes[1,0].scatter(dg['num_nodes'],cr,c=[COLORS['green'] if r>=0.8 else '#FF3B30' for r in cr],s=80,alpha=0.7,edgecolors='white')
axes[1,0].set_xlabel('Nodes'); axes[1,0].set_ylabel('Accuracy'); axes[1,0].set_title('Complexity vs Accuracy',fontweight='bold')
axes[1,0].axhline(y=0.8,color=COLORS['gray'],linestyle='--',alpha=0.4)
cfgr=dg['cfg_edges']/dg['num_edges'].replace(0,1); dfgr=dg['dfg_edges']/dg['num_edges'].replace(0,1)
axes[1,1].scatter(cfgr,dfgr,c=COLORS['accent'],s=dg['num_nodes']/3+20,alpha=0.6,edgecolors='white')
axes[1,1].set_xlabel('CFG Ratio'); axes[1,1].set_ylabel('DFG Ratio'); axes[1,1].set_title('CFG vs DFG Density',fontweight='bold')
plt.suptitle('Program Graph Analysis',fontsize=16,fontweight='bold',y=1.02); plt.tight_layout(); plt.show()
print(f"Total: {dg['num_nodes'].sum():,} nodes, {dg['num_edges'].sum():,} edges")
print(f"AST: {dg['ast_edges'].sum():,} ({dg['ast_edges'].sum()/dg['num_edges'].sum()*100:.1f}%), CFG: {dg['cfg_edges'].sum():,} ({dg['cfg_edges'].sum()/dg['num_edges'].sum()*100:.1f}%), DFG: {dg['dfg_edges'].sum():,} ({dg['dfg_edges'].sum()/dg['num_edges'].sum()*100:.1f}%)")

## 6. Confidence Calibration
Does 80% confidence actually mean 80% accuracy?

In [ ]:
calib = results.get('confidence_calibration',[])
if calib:
    dc = pd.DataFrame(calib)
    fig, axes = plt.subplots(1,2,figsize=(14,5.5))
    axes[0].plot([0,1],[0,1],'k--',alpha=0.4,label='Perfect'); axes[0].plot(dc['confidence_bin'],dc['accuracy'],'o-',color=COLORS['accent'],lw=2.5,ms=8,label='ACRS')
    axes[0].fill_between(dc['confidence_bin'],dc['accuracy'],dc['confidence_bin'],alpha=0.1,color=COLORS['accent'])
    axes[0].set_xlabel('Predicted Confidence'); axes[0].set_ylabel('Actual Accuracy'); axes[0].set_title('Calibration Curve',fontsize=14,fontweight='bold'); axes[0].legend()
    axes[0].set_xlim(-0.05,1.05); axes[0].set_ylim(-0.05,1.05)
    axes[1].bar(dc['confidence_bin'],dc['count'],width=0.08,color=COLORS['accent'],alpha=0.7,edgecolor='white')
    axes[1].set_xlabel('Confidence'); axes[1].set_ylabel('Count'); axes[1].set_title('Confidence Distribution',fontsize=14,fontweight='bold')
    plt.tight_layout(); plt.show()
    ece = np.mean([abs(r['accuracy']-r['confidence_bin'])*r['count'] for _,r in dc.iterrows()])/dc['count'].sum()
    print(f"Expected Calibration Error (ECE): {ece:.4f}")

## 7. Per-Sample Results Heatmap

In [ ]:
samples = results['per_sample_results']
sids = [s['id'].replace('_',' ').title() for s in samples]
fig, ax = plt.subplots(figsize=(10,10))
hd = np.array([[s['correct_detections'],s['missed_detections'],s['false_alarms']] for s in samples])
im = ax.imshow(hd,cmap='YlOrRd',aspect='auto',vmin=0,vmax=max(hd.max(),1))
for i in range(len(sids)):
    for j in range(3):
        ax.text(j,i,str(int(hd[i][j])),ha='center',va='center',fontsize=11,fontweight='bold',color='white' if hd[i][j]>3 else 'black')
ax.set_xticks([0,1,2]); ax.set_xticklabels(['Correct ✓','Missed ✗','False Alarms ⚡'],fontsize=11,fontweight='bold')
ax.set_yticks(range(len(sids))); ax.set_yticklabels(sids,fontsize=9)
ax.set_title('Per-Sample Detection Results',fontsize=15,fontweight='bold',pad=15)
plt.colorbar(im,ax=ax,shrink=0.6); plt.tight_layout(); plt.show()

## 8. Latency Analysis

In [ ]:
lats = [s['latency_ms'] for s in samples]; nds = [s['graph_nodes'] for s in samples]
fig, axes = plt.subplots(1,2,figsize=(14,5))
axes[0].scatter(nds,lats,c=COLORS['accent'],s=80,alpha=0.7,edgecolors='white')
z=np.polyfit(nds,lats,1); p=np.poly1d(z); xl=np.linspace(min(nds),max(nds),100)
axes[0].plot(xl,p(xl),'--',color='#FF3B30',alpha=0.6,label=f'slope={z[0]:.2f}ms/node'); axes[0].legend()
axes[0].set_xlabel('Nodes'); axes[0].set_ylabel('ms'); axes[0].set_title('Latency vs Complexity',fontweight='bold')
axes[1].hist(lats,bins=10,color=COLORS['accent'],edgecolor='white',alpha=0.8)
axes[1].axvline(np.mean(lats),color='#FF3B30',linestyle='--',label=f'Mean: {np.mean(lats):.1f}ms')
axes[1].axvline(np.median(lats),color='#34C759',linestyle='--',label=f'Median: {np.median(lats):.1f}ms')
axes[1].legend(); axes[1].set_xlabel('ms'); axes[1].set_title('Latency Distribution',fontweight='bold')
plt.tight_layout(); plt.show()
print(f"Mean: {np.mean(lats):.1f}ms, Median: {np.median(lats):.1f}ms, P95: {np.percentile(lats,95):.1f}ms")

## 9. Summary

| Metric | Value |
|--------|-------|
| **Micro F1** | 81.8% |
| **Macro F1** | 84.8% |
| **Recall** | 90.0% |
| **Precision** | 75.0% |
| **Samples** | 20 |
| **Patterns** | 6 |

### Key Findings
1. **High complexity, long params, god functions** — 100% F1, structural GAT analysis excels here
2. **Deep nesting** — 85.7% F1, AST depth features work well
3. **Missing return** — 60% F1, CFG path analysis needs reachability improvement
4. **Unused variables** — 60% F1, DFG def-use chains need scope refinement
5. **Clean code** — 92.3% F1, very few false positives on well-structured code

---
*ACRS Benchmark Suite v1.0*